# Enunciado da ponderada

**Objetivo:** Otimizar um modelo de rede neural pré-treinado para detecção de fraudes em cartões de crédito. Aplicar técnicas avançadas de ajuste fino de hiperparâmetros, como grid search e random search, com o objetivo de aprimorar as métricas de desempenho do modelo, incluindo precisão, recall, F1-score e AUC-ROC. A atividade também exige uma comparação entre o modelo otimizado e o modelo original, permitindo avaliar o impacto das modificações nos hiperparâmetros sobre o desempenho geral.

Para realizar esta atividade, você deve começar treinando o modelo de rede neural para detecção de fraudes em cartões de crédito. Os dados do cartão podem ser encontrados no link de conteúdo. Depois, obtenha as métricas de desempenho deste modelo como a precisão, recall, F1-score e AUC-ROC.  


Em seguida, defina uma faixa de valores para os hiperparâmetros que deseja otimizar. Aplique técnicas de ajuste fino de hiperparâmetros para melhorar o desempenho do modelo. Você pode usar métodos como grid search e random search para encontrar as melhores combinações de hiperparâmetros.


Após otimizar o modelo, compare os resultados obtidos com os resultados do modelo original. Analise como as mudanças nos hiperparâmetros impactaram o desempenho, considerando cada uma das métricas mencionadas. Por fim, documente todas as etapas realizadas e as observações feitas durante o processo.


Entregue o link do caderno `.ipynb` em um repositório GitHub.

## Importando os dados

In [8]:
%pip install gdown
import gdown

In [9]:
arquivo_destino_colab = "dataset.csv"
doc_id = "1u_OWAPkIdgJw1ah5xP_dGBFMSANxjxEl"
URL = f"https://drive.google.com/uc?id={doc_id}"
gdown.download(URL, arquivo_destino_colab, quiet=False)

Downloading...
From (original): https://drive.google.com/uc?id=1u_OWAPkIdgJw1ah5xP_dGBFMSANxjxEl
From (redirected): https://drive.google.com/uc?id=1u_OWAPkIdgJw1ah5xP_dGBFMSANxjxEl&confirm=t&uuid=283ece19-c9f4-4902-b0e0-dc71e03098b3
To: /content/dataset.csv
100%|██████████| 151M/151M [00:02<00:00, 56.3MB/s]


'dataset.csv'

## Treinamento do modelo inicial

In [10]:
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, roc_auc_score

import pandas as pd

df = pd.read_csv('dataset.csv')
X = df.drop('Class', axis=1)
y = df['Class']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

model_original = MLPClassifier(random_state=42)
model_original.fit(X_train, y_train)
y_pred = model_original.predict(X_test)
y_pred_prob = model_original.predict_proba(X_test)[:,1]
print(classification_report(y_test, y_pred))
print("AUC-ROC:", roc_auc_score(y_test, y_pred_prob))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00     85295
           1       0.44      0.68      0.54       148

    accuracy                           1.00     85443
   macro avg       0.72      0.84      0.77     85443
weighted avg       1.00      1.00      1.00     85443

AUC-ROC: 0.881918635324462


### Avaliação do Modelo Inicial

Após o treinamento inicial do modelo de rede neural para detecção de fraudes em cartões de crédito, obti as seguintes métricas de desempenho no conjunto de teste:

| Classe | Precision | Recall | F1-score | Suporte |
|--------|-----------|--------|----------|---------|
| 0 (não fraude) | 1.00      | 1.00   | 1.00     | 85.295  |
| 1 (fraude)     | 0.44      | 0.68   | 0.54     | 148     |

- **Acurácia total:** 1.00
- **Média macro:** Precision = 0.72, Recall = 0.84, F1-score = 0.77
- **Média ponderada:** Precision = 1.00, Recall = 1.00, F1-score = 1.00
- **AUC-ROC:** 0.8819

### Interpretação dos Resultados

- O modelo apresenta **alta acurácia global (1.00)**, mas essa métrica pode não ser precisa devido ao forte desbalanceamento das classes (muito mais transações legítimas do que fraudulentas).
- Para a classe de fraudes (classe 1), o modelo consegue identificar **68% das fraudes verdadeiras (recall = 0.68)**, porém a **precisão é mais baixa (0.44)**, indicando que cerca de 56% das transações sinalizadas como fraudes são falsos positivos.
- O F1-score de 0.54 para fraudes mostra que há um equilíbrio moderado entre precisão e recall.
- A métrica **AUC-ROC de 0.88** indica que o modelo tem uma boa capacidade geral de distinguir entre transações fraudulentas e legítimas.

## Definição dos hiperparâmetros

In [11]:
param_grid_optimized = {
    'hidden_layer_sizes': [(30,), (60,), (30,30)],
    'activation': ['relu', 'tanh'],
    'solver': ['adam'],
    'alpha': [0.0001, 0.001],
    'learning_rate_init': [0.001, 0.01]
}

### Definição dos hiperparâmetros para otimização

Para otimizar o desempenho do modelo de rede neural, defini um conjunto de hiperparâmetros para ajuste fino, com o objetivo de equilibrar abrangência da busca e tempo computacional. Os hiperparâmetros escolhidos e seus valores são:

- **hidden_layer_sizes:**  
  Define a arquitetura da rede em termos do número e tamanho das camadas ocultas. As opções consideradas foram:  
  - `(30,)`: uma camada oculta com 30 neurônios  
  - `(60,)`: uma camada oculta com 60 neurônios  
  - `(30, 30)`: duas camadas ocultas, cada uma com 30 neurônios  
  Essa escolha busca testar diferentes profundidades e larguras para o modelo.

- **activation:**  
  Função de ativação dos neurônios nas camadas ocultas, com opções:  
  - `'relu'` (função linear retificada), que é eficiente em redes profundas  
  - `'tanh'` (hiperbólica tangente), que pode ajudar em tamanhos menores

- **solver:**  
  Algoritmo para otimização do gradiente durante o treino.  
  Fixado em `'adam'` por ser um método adaptativo rápido e com bom desempenho, eliminando o custo de testar múltiplos solvers.

- **alpha:**  
  Parâmetro de regularização L2 que previne overfitting pela penalização dos pesos. Valores considerados:  
  - `0.0001` e `0.001`, para controle leve a moderado da complexidade do modelo.

- **learning_rate_init:**  
  Taxa inicial de aprendizado para o otimizador, com valores:  
  - `0.001` e `0.01` para testar a velocidade de atualização dos pesos e estabilidade do treino.




## Ajuste fino dos hiperparâmetros

In [13]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.neural_network import MLPClassifier

base_model = MLPClassifier(random_state=42, early_stopping=True, n_iter_no_change=10, validation_fraction=0.1, max_iter=200)

random_search = RandomizedSearchCV(
    base_model,
    param_distributions=param_grid_optimized,
    n_iter=10,
    cv=3,
    scoring='f1',
    verbose=2,
    n_jobs=-1,
    random_state=42
)
random_search.fit(X_train, y_train)
print("Melhores hiperparâmetros Random Search:", random_search.best_params_)
print("Melhor F1:", random_search.best_score_)

Fitting 3 folds for each of 10 candidates, totalling 30 fits
Melhores hiperparâmetros Random Search: {'solver': 'adam', 'learning_rate_init': 0.001, 'hidden_layer_sizes': (30,), 'alpha': 0.0001, 'activation': 'relu'}
Melhor F1: 0.6257981246389019


In [16]:
from sklearn.model_selection import GridSearchCV

grid_search = GridSearchCV(
    base_model,
    param_grid_optimized,
    cv=3,
    scoring='f1',
    verbose=2,
    n_jobs=-1
)
grid_search.fit(X_train, y_train)
print("Melhores hiperparâmetros Grid Search:", grid_search.best_params_)
print("Melhor F1:", grid_search.best_score_)

Fitting 3 folds for each of 24 candidates, totalling 72 fits
Melhores hiperparâmetros Grid Search: {'activation': 'relu', 'alpha': 0.0001, 'hidden_layer_sizes': (30,), 'learning_rate_init': 0.001, 'solver': 'adam'}
Melhor F1: 0.6257981246389019


## Avaliação do modelo otimizado

A avaliação do modelo otimizado usando o relatório de classificação e a métrica AUC-ROC indica o seguinte:

- **Precisão (Precision):** Para a classe minoritária (fraudes, classe 1), a precisão é 0.44, indicando que 44% das fraudes detectadas realmente são fraudes. Para a classe majoritária (não fraudes), a precisão é quase perfeita (1.00).
- **Recall:** O recall para fraudes é 0.68, ou seja, o modelo identifica corretamente 68% das fraudes reais, um valor razoável.
- **F1-score:** Métrica que combina precisão e recall, com valor 0.54 para fraudes e média ponderada global de 1.00 para os dados completos. O modelo apresenta um F1-score médio de 0.77 (macro), refletindo melhor desempenho para a classe majoritária.
- **Acurácia (Accuracy):** Muito alta (quase 1.00), mas deve ser interpretada com cautela devido ao forte desbalanceamento, que favorece a classe majoritária.
- **AUC-ROC:** O valor de 0.88 indica uma boa capacidade do modelo em distinguir entre classes fraudadoras e não fraudadoras.


## Avaliação do Grid Search


- Durante o Grid Search, foram avaliados 24 candidatos com 3 folds cada, totalizando 72 fits.
- Os melhores hiperparâmetros encontrados foram:  
  `{'activation': 'relu', 'alpha': 0.0001, 'hidden_layer_sizes': (30,), 'learning_rate_init': 0.001, 'solver': 'adam'}`.
- O melhor F1-score obtido na validação cruzada foi 0.63, correspondente a um equilíbrio semelhante ao encontrado no Random Search.


## Melhores Hiperparâmetros Encontrados pelo Random Search


- Solver: `adam`
- Learning rate: `0.001`
- Camadas ocultas: `(30,)`
- Regularização (alpha): `0.0001`
- Função de ativação: `relu`


## Análise


- O ajuste dos hiperparâmetros levou a um aumento no F1-score para aproximadamente 0.63 na validação cruzada, o que conduz a um balanço melhor entre precisão e recall do que um modelo padrão.
- Dados altamente desbalanceados garantem que métricas como recall e F1 para a classe minoritária sejam mais relevantes que accuracy.
- A AUC-ROC de 0.88 é sólida, denotando boa discriminação das fraudes.
- Ainda há espaço para melhorar a detecção da classe fraudulenta, potencialmente com técnicas adicionais (balanceamento de classes, Ensemble, etc.).


Essa avaliação comprova que o tuning aprimorou o desempenho do modelo na fração importante de dados (fraudes) reforçando a escolha de hiperparâmetros otimizados.